# Week 2: Long Context Processing (Apple 10-K)
Demonstrating detailed token analysis and M&A footnote extraction using Gemini 2.5 Flash's free context tier.

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.ingestion.pdf_loader import load_pdf
from src.ingestion.tokenizer_analysis import TokenAnalyzer

# Load document
pdf_path = "../data/raw/apple10k.pdf"
document = load_pdf(pdf_path)

# Analyze tokens
analyzer = TokenAnalyzer()
document = analyzer.analyze_document(document)

# High-level document summary
summary = analyzer.document_summary(document)
print("=== Document Token Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

=== Document Token Summary ===
  total_chars: 273800
  total_tokens: 60566
  total_paragraphs: 4790
  avg_tokens_per_para: 11.95
  median_tokens_per_para: 5.0
  max_tokens_para: 71
  min_tokens_para: 1
  std_dev_tokens: 12.31
  p90_tokens: 30.0
  paragraphs_over_500_tokens: 0
  paragraphs_over_1000_tokens: 0


In [2]:
# Per-paragraph token statistics
df = analyzer.paragraph_statistics(document)

print(f"Total paragraphs: {len(df)}")
print(df.describe())
df.head(10)

Total paragraphs: 4790
       paragraph_id  char_count       tokens  token_density  cumulative_tokens
count   4790.000000  4790.00000  4790.000000    4790.000000        4790.000000
mean    2394.500000    55.98977    11.949061       0.408914       28674.315031
std     1382.898225    63.74328    12.310451       0.286334       14034.591846
min        0.000000     1.00000     1.000000       0.090900           3.000000
25%     1197.250000     5.00000     3.000000       0.183700       20173.750000
50%     2394.500000    17.00000     5.000000       0.272700       28170.500000
75%     3591.750000   140.00000    25.000000       0.500000       38044.500000
max     4789.000000   189.00000    71.000000       2.000000       57236.000000


,paragraph_id,char_count,tokens,token_density,cumulative_tokens,preview
0,0,13,3,0.2308,3,UNITED STATES
1,1,34,8,0.2353,11,SECURITIES AND EXCHANGE COMMISSION
2,2,22,8,0.3636,19,"Washington, D.C. 20549"
3,3,9,4,0.4444,23,FORM 10-K
4,4,10,4,0.4000,27,(Mark One)
5,5,86,30,0.3488,57,☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d...
6,6,44,12,0.2727,69,"For the fiscal year ended September 27, 2025"
7,7,2,1,0.5000,70,or
8,8,90,29,0.3222,99,☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR ...
9,9,60,14,0.2333,113,For the transition period from to...


In [4]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.llm import gemini_client
from src.prompts import long_context_prompts

prompt = long_context_prompts.footnote_extraction_prompt(document.text)
# Uncomment to call the API (uses ~60k tokens)
response = gemini_client.query(prompt)
print(response)

Here are the findings from the 10-K document:

**Topic**: Contingent Liabilities & Pending Litigations - European Commission Digital Markets Act Investigations
- **Clause/Footnote Excerpt**: "On March 25, 2024, the Commission announced that it had opened a formal noncompliance investigation against the Company under Article 5(4) of the EU DMA... On April 23, 2025, the Commission fined the Company €500 million in the Article 5(4) Investigation and issued a cease and desist order requiring the Company to remove technical and commercial restrictions that prevent developers from steering users to alternative distribution channels outside the App Store. The Company has appealed the Commission’s Article 5(4) decision. Also on April 23, 2025, the Commission issued preliminary findings in the Article 6(4) Investigation. If the Commission makes a final determination... it can issue a cease and desist order and may impose fines up to 10% of the Company’s annual worldwide net sales." (Item 3. Leg